# 🔥 Yangın & Duman Tespiti — Colab Eğitim Notebook'u

Bu notebook projenin tüm adımlarını sırayla çalıştırır:

1. Ortam kurulumu (GPU, Drive, paketler)
2. Veri setini indirme (Colab sunucusuna iner, **senin internetini kullanmaz**)
3. Veri ön işleme + augmentation
4. Üç modelin eğitimi: **YOLO11**, **YOLOv8**, **RT-DETR**
5. Test setinde karşılaştırma
6. Tahmin örnekleri ve Gradio demosu

**Başlamadan önce:** `Runtime > Change runtime type` menüsünden GPU seç (Pro'da L4 veya A100 önerilir).

**Ön koşul:** Proje klasörünü (`configs/`, `src/`, `notebooks/`, `requirements.txt`) Google Drive'ına yükle, örn. `MyDrive/fire_smoke_detection/`.

In [ ]:
!nvidia-smi

## 1. Drive ve proje klasörü

- Kod ve **eğitim çıktıları** Drive'da durur → oturum kopsa bile kaybolmaz.
- **Veri** `/content/data` altında (Colab'ın yerel diski) durur → binlerce küçük dosyayı Drive'dan okumak çok yavaş olurdu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/fire_smoke_detection'  # projeyi yüklediğin yer
os.chdir(PROJECT_DIR)
os.environ['FSD_DATA_DIR'] = '/content/data'
print(sorted(os.listdir()))

In [ ]:
!pip install -q -r requirements.txt
import ultralytics, albumentations
print('ultralytics', ultralytics.__version__, '| albumentations', albumentations.__version__)

## 2-3. Veri indirme + ön işleme

İlk çalıştırmada veri seti Kaggle'dan indirilir (~3 GB, 1-3 dk), alt küme seçilir, temizlenir, küçültülür ve augmentation uygulanır. Sonuç Drive'a zip olarak önbelleğe alınır; sonraki oturumlarda sadece zip açılır (~30 sn).

> `configs/config.yaml` içinde `dataset` ayarlarını değiştirirsen `data_cache/processed.zip` dosyasını sil ki veri yeniden hazırlansın.

In [ ]:
CACHE_ZIP = f'{PROJECT_DIR}/data_cache/processed.zip'

if os.path.exists(CACHE_ZIP):
    !mkdir -p /content/data && unzip -q -o "{CACHE_ZIP}" -d /content/data
    print('İşlenmiş veri Drive önbelleğinden geri yüklendi.')
else:
    !python -m src.download_data
    !python -m src.prepare_data
    !mkdir -p "{PROJECT_DIR}/data_cache" && cd /content/data && zip -q -r "{CACHE_ZIP}" processed
    print('İşlenmiş veri Drive\'a önbelleklendi.')

In [ ]:
from IPython.display import Image, Markdown, display
display(Image('outputs/reports/dataset_stats.png'))
display(Image('outputs/reports/augmentation_preview.jpg', width=700))

## 4. Eğitim

Her model ayrı hücrede. Varsayılan ayarlarla (4000 train + ~1200 augmented görsel, 60 epoch, erken durdurma açık) tahmini süreler:

| Model | T4 | L4 / A100 |
|---|---|---|
| YOLO11s | ~1.5-2 sa | ~40-60 dk |
| YOLOv8s | ~1.5-2 sa | ~40-60 dk |
| RT-DETR-l | ~4-5 sa | ~1.5-2.5 sa |

Önce her şeyin çalıştığını görmek için `--epochs 3` ile kısa bir deneme yapabilirsin.

**Oturum koparsa:** notebook'u baştan çalıştır (Drive + kurulum + veri hücreleri), sonra ilgili modeli `--resume` ile başlat.

In [ ]:
!python -m src.train --model yolo11

In [ ]:
!python -m src.train --model yolov8

In [ ]:
# T4'te bellek yetmezse: --batch 8
!python -m src.train --model rtdetr

In [ ]:
# Kopan eğitime devam etmek için (ilgili satırın başındaki # işaretini kaldır):
# !python -m src.train --model rtdetr --resume

## 5. Test setinde karşılaştırma

In [ ]:
!python -m src.evaluate
display(Image('outputs/reports/model_comparison.png'))
display(Markdown(open('outputs/reports/model_comparison.md', encoding='utf-8').read()))

## 6. Tahmin örnekleri

In [ ]:
import glob, random, shutil
random.seed(0)
samples = random.sample(sorted(glob.glob('/content/data/processed/images/test/*.jpg')), 6)
os.makedirs('/content/samples', exist_ok=True)
for p in samples:
    shutil.copy(p, '/content/samples/')

MODEL = 'yolo11'  # yolo11 | yolov8 | rtdetr
!python -m src.predict --model {MODEL} --source /content/samples

for p in sorted(glob.glob(f'outputs/predictions/{MODEL}/*_pred.jpg'))[:6]:
    display(Image(p, width=600))

## 7. Gradio demosu

Çıktıdaki `https://....gradio.live` linkine tıkla (72 saat geçerli). Durdurmak için hücreyi durdur (■).

In [ ]:
!python -m src.app --share